# WEL01 · M01 입력 자료의 형식과 오류 검출 확인 (교수자·해설용)

- 본 노트북은 **미션1(M01) 전용 교수자·해설용** 노트북입니다 (`ROLE = 'solution'`).
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일 없이 위에서부터 바로 실행할 수 있습니다.
- 각 STEP 코드 셀에는 모범 pandas 풀이, 경계 조건 예외 처리, 해설 주석이 포함되어 있습니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='solution'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M01 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M01"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M01 입력 자료의 형식과 오류 검출 확인

건강지원반은 센서 자료를 경보 프로그램에 넣기 전에 정해진 형식인지 확인해야 합니다. 이번 시험에는 정상 형식의 자료 20개와 오류를 넣은 자료 5개가 들어 있습니다. 정상 자료를 거부하지 않고 오류 자료를 찾아냈는지 각 행에서 확인한 뒤, 다시 살펴볼 자료와 그 이유를 정리하세요.

실습 시간 90분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M01'
PARAMETERS = {}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M01 STEP 01 — 정상 자료와 오류 시험 자료 구분하기

<h4>이 단계에서 해결할 일</h4><p>건강지원 프로그램에는 정상적인 형식의 자료 20개와 일부러 오류를 넣은 시험 자료 5개가 함께 들어 있습니다. 먼저 두 종류를 구분해야 나중에 정상 자료를 잘못 거부했는지, 오류 자료를 놓쳤는지 따로 확인할 수 있습니다. 이 단계에서는 센서 수치를 판단하지 않고 각 자료가 어떤 시험에 쓰이는지만 표시합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m01_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>자료_유형</code>이며 배정 시간은 18분입니다. 노트북의 <code>M01 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>case_kind</code> — 자료를 만든 목적입니다. 정상 자료에는 정상 구조, 오류를 넣은 자료에는 오류 주입이라고 적혀 있습니다.</li><li><code>원행ID</code> — 입력 표의 각 행에 붙인 번호입니다. 계산 전후 같은 자료를 찾는 데 사용합니다.</li></ul><h4>수행 순서</h4><ol><li>입력 표 raw에서 case_kind 열을 찾고 고유한 값과 각 값의 개수를 확인합니다.</li><li>case_kind의 값을 문자열로 읽고 앞뒤에 붙은 공백만 제거합니다. 중간 공백이나 원래 문구는 바꾸지 않습니다.</li><li>정리한 값을 새 열 자료_유형에 넣습니다. case_kind를 덮어쓰지 않고 결과 표를 step1에 저장합니다.</li><li>정상 구조 20행, 오류 주입 5행, 전체 25행이 남아 있는지 확인합니다. 행 순서와 원행ID는 입력 표와 같아야 합니다.</li></ol><h4>결과를 읽는 방법</h4><p>자료_유형은 자료의 용도를 나타내는 문자열입니다. 정상 구조라는 표시가 있다고 해서 해당 자료가 검증을 통과한 것은 아닙니다. 실제 오류 유무는 다음 단계에서 확인합니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"case_kind": "  정상 구조  "}</code></td><td><code>정상 구조</code></td><td>앞뒤 공백만 없어집니다.</td></tr><tr><td><code>{"case_kind": "오류 주입"}</code></td><td><code>오류 주입</code></td><td>오류 시험 자료라는 구분을 유지합니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>문자열을 읽을 수 없거나 두 범주 이외의 값이 나오면 임의로 정상 구조로 바꾸지 말고 해당 원행ID와 원래 값을 기록해 교관에게 확인을 요청합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: case_kind 대신 센서값으로 자료 유형을 정하거나, 오류 시험 자료를 삭제하거나, 모든 행에 같은 문구를 넣으면 FAIL입니다.

In [ ]:
# M01 STEP 01 — 정상 자료와 오류 시험 자료 구분하기 (교수자·해설용 예시 풀이)
# 해설: case_kind의 앞뒤 공백만 제거하고, 중간 공백이나 원래 문구는 바꾸지 않습니다.
# 범주('정상 구조', '오류 주입') 외의 값이나 결측값이 나오면 임의로 채우지 않고 예외로 알립니다.
step1 = raw.copy()

def _clean_case_kind(val):
    if pd.isna(val) or not isinstance(val, str):
        raise ValueError(f"읽을 수 없는 case_kind 값: {val!r}")
    cleaned = val.strip()
    if cleaned not in ("정상 구조", "오류 주입"):
        raise ValueError(f"허용되지 않은 case_kind 범주: {val!r}")
    return cleaned

step1["자료_유형"] = step1["case_kind"].map(_clean_case_kind)
display(step1[["원행ID", "자료_유형"]].head(10))

### M01 STEP 02 — 자료별로 발견된 오류 개수 세기

<h4>이 단계에서 해결할 일</h4><p>검증 프로그램은 각 자료에서 발견한 오류를 목록으로 남깁니다. 이 목록이 비어 있으면 발견된 오류가 없다는 뜻입니다. 오류 메시지의 글자 수를 세면 오류 개수와 전혀 다른 값이 나오므로, 먼저 저장 형식을 읽은 뒤 목록의 항목 수를 세어야 합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m01_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>구조_오류수</code>이며 배정 시간은 18분입니다. 노트북의 <code>M01 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>validation_errors</code> — 오류 메시지 목록을 JSON 형식의 문자열로 저장한 열입니다. []는 오류가 없다는 뜻이고 ["풍속 누락"]은 오류 1개를 뜻합니다.</li></ul><h4>수행 순서</h4><ol><li>앞 단계 결과 표 step1에서 validation_errors 값을 확인합니다. 대괄호로 감싼 오류 목록인지 봅니다.</li><li>각 문자열을 json.loads 등으로 읽어 Python의 목록으로 바꿉니다. 문자열 그대로 len을 적용하지 않습니다.</li><li>목록에 들어 있는 메시지 개수를 세어 구조_오류수 열에 저장합니다. 값은 0 이상의 정수여야 합니다.</li><li>기존 열과 자료_유형 열을 모두 보존한 표를 step2에 저장합니다. 정상 자료 1개와 오류 자료 1개의 원문 목록·계산한 개수를 나란히 확인합니다.</li></ol><h4>결과를 읽는 방법</h4><p>구조_오류수는 검증기가 한 자료에서 발견한 오류 메시지의 개수입니다. 오류 0개는 아직 발견된 구조 오류가 없다는 뜻이며, 실제 병사의 건강이 정상이라는 뜻이 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"validation_errors": "[]"}</code></td><td><code>0</code></td><td>빈 목록에는 오류 메시지가 없습니다.</td></tr><tr><td><code>{"validation_errors": "[\"풍속 누락\", \"기기 종류 오류\"]"}</code></td><td><code>2</code></td><td>메시지가 두 개이므로 2입니다. 글자 수가 아닙니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>빈 문자열이나 잘못된 JSON은 []와 다릅니다. 읽을 수 없는 값을 오류 0개로 대신 채우지 말고, 계산을 중단한 위치와 원문을 기록해 교관에게 확인받습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 문자열 길이를 오류 개수로 쓰거나, JSON을 읽지 못한 값을 0으로 처리하거나, 목록 항목 수와 다른 값을 저장하면 FAIL입니다.

In [ ]:
# M01 STEP 02 — 자료별로 발견된 오류 개수 세기 (교수자·해설용 예시 풀이)
# 해설: validation_errors는 JSON 배열 문자열이므로 json.loads로 파싱한 뒤 목록의 길이를 셉니다.
# 문자열 자체의 len()을 쓰거나 파싱 실패를 0으로 채우지 않습니다.
step2 = step1.copy()

def _count_errors(val):
    if pd.isna(val) or not isinstance(val, str):
        raise ValueError(f"validation_errors 결측/형식 오류: {val!r}")
    parsed = json.loads(val)
    if not isinstance(parsed, list):
        raise ValueError(f"JSON 목록이 아닙니다: {val!r}")
    return len(parsed)

step2["구조_오류수"] = step2["validation_errors"].map(_count_errors).astype(int)
display(step2[["원행ID", "구조_오류수"]].head(10))

### M01 STEP 03 — 구조 검증을 통과한 자료 표시하기

<h4>이 단계에서 해결할 일</h4><p>지원 프로그램에 사용할 수 있는 자료를 고르려면 앞 단계에서 센 오류 개수를 통과 여부로 바꿔야 합니다. 여기서 통과는 형식 검사가 끝났다는 뜻입니다. 오류가 하나라도 남은 자료를 통과로 표시하면 잘못된 입력이 다음 계산으로 넘어갈 수 있습니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m01_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>검증_통과</code>이며 배정 시간은 18분입니다. 노트북의 <code>M01 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>구조_오류수</code> — 앞 단계에서 오류 목록의 항목 수를 센 값입니다. 0이면 발견된 오류가 없고 1 이상이면 구조 오류가 남아 있습니다.</li></ul><h4>수행 순서</h4><ol><li>step2의 구조_오류수 열이 비어 있지 않고 0 이상의 정수값인지 확인합니다.</li><li>구조_오류수가 정확히 0인 행에만 True를, 1 이상인 행에는 False를 부여합니다.</li><li>결과를 검증_통과 열로 추가하고 step3에 저장합니다. True와 False는 따옴표가 붙은 글자가 아니라 참·거짓 값이어야 합니다.</li><li>오류 0개인 행과 오류가 있는 행을 각각 골라 구조_오류수와 검증_통과가 맞게 연결되는지 확인합니다.</li></ol><h4>결과를 읽는 방법</h4><p>검증_통과=True는 이 실습의 구조 검사를 통과했다는 뜻이고 False는 입력 내용을 고쳐야 한다는 뜻입니다. True를 건강 정상, False를 건강 위험으로 해석하지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"구조_오류수": 0}</code></td><td><code>True</code></td><td>발견된 구조 오류가 없으므로 통과입니다.</td></tr><tr><td><code>{"구조_오류수": 1}</code></td><td><code>False</code></td><td>오류가 하나라도 남으면 통과시키지 않습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>오류 개수가 비어 있거나 음수라면 이전 단계 결과부터 확인합니다. 알 수 없는 값을 오류 0개처럼 취급하지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 오류 1개를 허용하거나, 0과 1의 의미를 반대로 적용하거나, 입력이 비었는데 True로 채우면 FAIL입니다.

In [ ]:
# M01 STEP 03 — 구조 검증을 통과한 자료 표시하기 (교수자·해설용 예시 풀이)
# 해설: 구조_오류수가 비어 있지 않고 0 이상 정수인지 확인한 뒤, 0인 행만 실제 불리언 True로 표시합니다.
step3 = step2.copy()
if step3["구조_오류수"].isna().any() or (step3["구조_오류수"] < 0).any():
    raise ValueError("구조_오류수에 결측 또는 음수가 있습니다.")
step3["검증_통과"] = (step3["구조_오류수"] == 0).astype(bool)
display(step3[["원행ID", "검증_통과"]].head(10))

### M01 STEP 04 — 예상한 검증 결과와 실제 결과 비교하기

<h4>이 단계에서 해결할 일</h4><p>오류를 일부러 넣은 자료는 거부되어야 정상적으로 시험을 통과한 것입니다. 따라서 검증_통과가 False라고 해서 그 시험이 실패한 것은 아닙니다. 이 단계에서는 자료를 만들 때 정한 기대 결과와 검증기의 실제 결과가 같은지 확인합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m01_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>검출_기대일치</code>이며 배정 시간은 18분입니다. 노트북의 <code>M01 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>검증_통과</code> — 실제로 구조 검사를 통과했는지 나타내는 앞 단계의 참·거짓 값입니다.</li><li><code>expected_valid</code> — 시험을 설계할 때 정한 기대 결과입니다. 정상 자료는 True, 오류를 주입한 자료는 False입니다.</li></ul><h4>수행 순서</h4><ol><li>step3에서 두 열이 모두 참·거짓 값으로 들어 있는지 확인합니다.</li><li>같은 행의 검증_통과와 expected_valid를 비교합니다. 둘 다 True이거나 둘 다 False일 때 결과를 True로 만듭니다.</li><li>두 값이 다르면 False로 표시하고 검출_기대일치 열에 저장합니다. 이 열을 추가한 표를 step4에 저장합니다.</li><li>정상 자료 통과, 오류 자료 거부, 정상 자료 오거부, 오류 자료 통과의 네 가지 경우를 아래 예와 비교합니다. 실제 입력에 없는 경우는 가정 사례로만 표시합니다.</li></ol><h4>결과를 읽는 방법</h4><p>검출_기대일치=True는 검증기가 시험 의도대로 작동했다는 뜻입니다. 특히 expected_valid=False와 검증_통과=False의 조합은 올바른 오류 검출이므로 True입니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"검증_통과": true, "expected_valid": true}</code></td><td><code>True</code></td><td>정상 자료가 통과했습니다.</td></tr><tr><td><code>{"검증_통과": false, "expected_valid": false}</code></td><td><code>True</code></td><td>오류 자료를 예상대로 거부했습니다.</td></tr><tr><td><code>{"검증_통과": true, "expected_valid": false}</code></td><td><code>False</code></td><td>오류 자료를 놓쳤습니다.</td></tr><tr><td><code>{"검증_통과": false, "expected_valid": true}</code></td><td><code>False</code></td><td>정상 자료를 잘못 거부했습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>오류 자료가 거부되었다는 이유로 검출_기대일치를 False로 두지 않습니다. 기대 결과가 비어 있으면 시험의 성공 여부를 정할 수 없으므로 확인을 요청합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 실제 통과 여부만 복사하거나, 두 값이 모두 False인 정상적인 오류 검출을 실패로 처리하면 FAIL입니다.

In [ ]:
# M01 STEP 04 — 예상한 검증 결과와 실제 결과 비교하기 (교수자·해설용 예시 풀이)
# 해설: expected_valid == False이고 검증_통과 == False인 오류 주입 거부는 올바른 검출(True)입니다.
step4 = step3.copy()
if step4["검증_통과"].isna().any() or step4["expected_valid"].isna().any():
    raise ValueError("검증_통과 또는 expected_valid에 결측값이 있습니다.")
step4["검출_기대일치"] = (step4["검증_통과"].astype(bool) == step4["expected_valid"].astype(bool)).astype(bool)
display(step4[["원행ID", "검출_기대일치"]].head(10))

### M01 STEP 05 — 자료 규격을 다시 확인해야 할 행 고르기

<h4>이 단계에서 해결할 일</h4><p>검증기가 시험 의도와 다르게 작동했거나, 프로그램이 사용하기로 한 필드가 빠져 있으면 자료 규격을 다시 살펴야 합니다. 이 단계에서는 해당 행을 찾아 교관 또는 데이터 담당자가 확인할 수 있도록 표시합니다. 오류를 넣은 시험 자료라는 이유만으로 모두 보완 대상으로 지정하는 것은 아닙니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m01_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>명세_검토대상</code>이며 배정 시간은 18분입니다. 노트북의 <code>M01 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>검출_기대일치</code> — 검증기가 기대한 대로 통과 또는 거부했는지 나타내는 값입니다.</li><li><code>field_count</code> — 명세서에 정의한 세부 필드 개수입니다. 이번 실습의 기준은 17개입니다. 원본 표의 열 개수와 다릅니다.</li></ul><h4>수행 순서</h4><ol><li>step4의 검출_기대일치와 field_count를 확인합니다. field_count는 표에 열이 몇 개인지 세어 대신 쓰지 않습니다.</li><li>검출_기대일치가 False이거나 field_count가 17이 아니면 명세_검토대상을 True로 표시합니다.</li><li>기대 결과가 일치하고 필드 수도 17이면 False로 표시합니다. 결과 표를 step5에 저장합니다.</li><li>True인 행의 원행ID·검증 오류 내용·field_count·확인할 사항을 기록합니다. True가 한 건도 없으면 0건이라고 쓰고 정상적으로 검출된 오류 시험 자료와 구분해 설명합니다.</li></ol><h4>결과를 읽는 방법</h4><p>명세_검토대상=True는 규격이나 검증 동작을 다시 확인해야 한다는 뜻입니다. False는 이 두 조건에 따른 추가 보완 대상이 없다는 뜻이며, 기상·기기 명세 전체를 자동으로 승인했다는 뜻은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"검출_기대일치": true, "field_count": 17}</code></td><td><code>False</code></td><td>예상한 검증 결과이며 필드 수도 기준과 같습니다.</td></tr><tr><td><code>{"검출_기대일치": false, "field_count": 17}</code></td><td><code>True</code></td><td>검증 동작을 다시 확인해야 합니다.</td></tr><tr><td><code>{"검출_기대일치": true, "field_count": 16}</code></td><td><code>True</code></td><td>정의된 필드가 기준 개수와 다릅니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>오류를 주입한 자료가 예상대로 거부되고 필드 수도 17개라면 False가 맞습니다. 기상·기기 세부 명세는 상황판의 명세 내용에서 별도로 확인합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 두 문제를 모두 만족할 때만 True로 만들거나, 정상적으로 거부된 오류 시험 자료를 무조건 보완 대상으로 만들면 FAIL입니다.

In [ ]:
# M01 STEP 05 — 자료 규격을 다시 확인해야 할 행 고르기 (교수자·해설용 예시 풀이)
# 해설: 검출_기대일치가 False이거나 field_count가 17이 아닌 경우(OR)만 재검토 대상으로 표시합니다.
step5 = step4.copy()
if step5["field_count"].isna().any():
    raise ValueError("field_count에 결측값이 있습니다.")
step5["명세_검토대상"] = ((~step5["검출_기대일치"]) | (step5["field_count"] != 17)).astype(bool)
display(step5[["원행ID", "명세_검토대상"]].head(10))

### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
